# Evidencias individuales de Docencia

Construye `data/evidencias/docencia/evidencias_docencia.csv` (`DOCENCIA_MATERIA` y `ACTIVIDAD_CARGA`).
Reglas en `README.md` y formato de atributos en `../ATRIBUTOS.md`. Requiere haber generado antes las evidencias de trayectoria.

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "02_evidencias").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from evidencias.docencia import construir_evidencias_docencia

pd.set_option("display.max_colwidth", 200)

In [ ]:
evidencias, reporte, validacion = construir_evidencias_docencia(guardar=True)
print(json.dumps(reporte, ensure_ascii=False, indent=2))

## Validación (todas las filas deben tener `n_fallas = 0`)

In [ ]:
validacion

## Ejemplos

In [ ]:
for tipo_id, grupo in evidencias.groupby("tipo_id"):
    print("\n=====", tipo_id)
    for _, r in grupo.sample(8, random_state=7).iterrows():
        print(f"- {r.evidencia_id} | {r.texto}")

## Materias dictadas en varios semestres, con cambio de código

In [ ]:
materias = evidencias[evidencias["tipo_id"] == "DOCENCIA_MATERIA"]
attrs = materias["atributos"].map(json.loads)
varias = materias[attrs.map(lambda a: a["n_periodos"] >= 4 and len(a["codigos"]) > 1)]
print(f"Evidencias dictadas en 4 o más semestres y con más de un código: {len(varias)}")
for _, r in varias.sample(5, random_state=3).iterrows():
    a = json.loads(r["atributos"])
    print()
    print("=" * 90)
    print("Texto:", r["texto"])
    for k in ["codigos", "componentes", "n_periodos", "periodos", "n_paralelos", "n_estudiantes_total", "es_compartida"]:
        print(f"  {k}: {a[k]}")
    print("  primer periodo:", json.dumps(a["detalle_periodos"][0], ensure_ascii=False))

## Actividades de la carga politécnica

`CON_OBJETO`: una evidencia por persona + función + tema. `RUTINA`: una evidencia por persona + función, con los temas (casi siempre la carrera) en una lista.

In [ ]:
act = evidencias[evidencias["tipo_id"] == "ACTIVIDAD_CARGA"]
attrs = act["atributos"].map(json.loads)
for clase in ["CON_OBJETO", "RUTINA"]:
    sub = act[attrs.map(lambda a: a["clase_funcion"] == clase)]
    print(f"\n===== {clase}: {len(sub)} evidencias")
    for _, r in sub.sample(6, random_state=5).iterrows():
        a = json.loads(r["atributos"])
        print(f"- {r.texto}")
        print(f"    años: {a['anios']} | actividades: {a['actividades']} | temas: {a['temas'][:3]} | personas con la misma función: {a['n_personas_misma_funcion']}")

## Registros no considerados (para revisión)

In [ ]:
nc = pd.read_csv(PROJECT_ROOT / "data" / "evidencias" / "docencia" / "registros_no_considerados.csv")
nc.groupby(["tipo_id", "motivo"]).size()